In [ ]:
# Bibliotecas de manipulação e visualização de dados
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Pré-processamento e modelagem
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Algoritmos de clusterização
from sklearn.cluster import KMeans  
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

# Fixando seed 
SEED = 42
np.random.seed(SEED)

***Carregamento dos Dados:***

In [ ]:
# Faz a leitura do dataset de transações com cartão de crédito
# Cada linha é uma transação; 'Class' indica fraude (1) ou legítima (0)
df = pd.read_csv('creditcard.csv')
print(df.shape)
df.head()

# Questão 1:

***1.Análise exploratória:***

In [ ]:
df.info()
df.describe()

***1.1.Análise de Classe:***

In [ ]:
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print("=" * 45)
print("DISTRIBUIÇÃO DAS CLASSES")
print("=" * 45)
print(f"  {'Classe':<12} {'Quantidade':>10} {'Porcentagem':>12}")
print("-" * 45)
for classe in class_counts.index:
    label = "Legítima (0)" if classe == 0 else "Fraude (1)"
    print(f"  {label:<12} {class_counts[classe]:>10} {class_pct[classe]:>11.3f}%")
print("=" * 45)

***1.2.Histogramas (Amount e Time):***

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Amount:
sns.histplot(df['Amount'], bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribuição - Amount')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequência')

# Time:
sns.histplot(df['Time'], bins=50, kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Distribuição - Time')
axes[1].set_xlabel('Time (segundos)')
axes[1].set_ylabel('Frequência')

plt.tight_layout()
plt.show()

***2.Valores ausentes:***

*2.1.Análise de Valores Ausentes:*

In [ ]:
ausentes = df.isnull().sum()
porcentagem = (ausentes / len(df)) * 100

resumo = pd.DataFrame({
    'Ausentes': ausentes,
    'Porcentagem (%)': porcentagem.round(2)
})

print(resumo[resumo['Ausentes'] > 0])

*2.2.Tratamento dos Valores Ausentes:*

In [ ]:
if ausentes.sum() > 0: # Preenche os valores ausentes com a média da coluna
    df.fillna(df.mean(), inplace=True)
    print("Valores ausentes encontrados e preenchidos com a média.")
    
else:
    print("Não há valores ausentes no dataset.")

***3.Redundância e inconsistência:***

*3.1.Detecção e remoção de registros duplicados(redundantes):*

In [ ]:
duplicados = df.duplicated().sum()
print('Duplicados:', duplicados)

print("Antes:", df.shape)
df.drop_duplicates(keep='first', inplace=True) # Remove registros idênticos, mantendo a primeira ocorrência
print("Depois:", df.shape)

*3.2.Detecção e remoção de registros inconsistentes:*

In [ ]:
df.describe().T

np.isinf(df).sum().sum() # como o csv possui é numerico, podemos verificar se há valores infinitos
print("Valores infinitos encontrados:", np.isinf(df).sum().sum())
df.replace([np.inf, -np.inf], np.nan, inplace=True)

***4.Análise de correlação e multicolinearidade:***

*4.1.Examinando as correlações:*

In [ ]:
correlacao = df.corr(numeric_only=True)

plt.figure(figsize=(12,8))

sns.heatmap(
    correlacao,
    cmap='coolwarm',
    center=0
)

plt.title('Matriz de Correlação')
plt.show()

#### Conclusão da Análise de Correlação

O heatmap confirma que:

- **Baixa correlação entre V1–V28:** como esperado para componentes de PCA,
  as variáveis são aproximadamente ortogonais entre si, o que elimina o problema
  de multicolinearidade nesse subconjunto de atributos.

- **Time e Amount:** apresentam correlações fracas com as demais variáveis,
  sem representar risco de multicolinearidade relevante para o modelo.

- **Class:** possui correlações fracas a moderadas com algumas componentes
  (especialmente V17, V14 e V12, com valores negativos), o que indica que
  nenhuma variável isolada é suficiente para detectar fraude, o modelo precisa
  combinar múltiplos atributos para classificar corretamente.

- **Ausência de multicolinearidade grave:** não há pares de variáveis preditoras
  com correlação próxima de 1 ou -1, o que é positivo para a estabilidade
  e interpretabilidade do modelo.

Em resumo, o heatmap confirma que o conjunto de dados está bem condicionado
para o treinamento, sem redundâncias expressivas entre os atributos.

***5.Divisão treino–teste estratificada:***

In [ ]:
X = df.drop('Class', axis=1)  # Features
y = df['Class']                # Alvo (fraude ou não)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,  # 20% teste, 80% treino
    stratify=y,     # mantém a proporção das classes
    random_state=SEED
)

print("=" * 45)
print("DIVISÃO TREINO — TESTE")
print("=" * 45)
print(f"  Total de amostras : {len(df)}")
print(f"  Treino            : {X_train.shape[0]}")
print(f"  Teste             : {X_test.shape[0]}")
print(f"  Features          : {X_train.shape[1]}")
print("-" * 45)
print(f"  Fraudes no treino : {y_train.sum()} ({y_train.mean()*100:.3f}%)")
print(f"  Fraudes no teste  : {y_test.sum()} ({y_test.mean()*100:.3f}%)")
print("=" * 45)

***6.Outliers:***

***6.1.Boxplot:***

***6.1.1.Boxplot de Amount:***

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=X_train['Amount'])
plt.title('Boxplot - Amount')
plt.show()

***6.1.2.Boxplot de Time:***

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=X_train['Time'])
plt.title('Boxplot - Time')
plt.show()

***6.2.Cálculo de outliers:***

***6.2.1.Cálculo de outliers em Amount:***

In [ ]:
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_amount = df[
    (df['Amount'] < limite_inferior) |
    (df['Amount'] > limite_superior)
]

print("=" * 45)
print("OUTLIERS — Amount")
print("=" * 45)
print(f"  Q1               : {Q1:.2f}")
print(f"  Q3               : {Q3:.2f}")
print(f"  IQR              : {IQR:.2f}")
print(f"  Limite inferior  : {limite_inferior:.2f}")
print(f"  Limite superior  : {limite_superior:.2f}")
print("-" * 45)
print(f"  Outliers encontrados : {len(outliers_amount)}")
print(f"  % do dataset         : {len(outliers_amount)/len(df)*100:.2f}%")
print("=" * 45)

***6.2.2.Cálculo de outliers em Time:***

In [ ]:
Q1 = df['Time'].quantile(0.25)
Q3 = df['Time'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_time = df[
    (df['Time'] < limite_inferior) |
    (df['Time'] > limite_superior)
]

print("=" * 45)
print("OUTLIERS — Time")
print("=" * 45)
print(f"  Q1               : {Q1:.2f}")
print(f"  Q3               : {Q3:.2f}")
print(f"  IQR              : {IQR:.2f}")
print(f"  Limite inferior  : {limite_inferior:.2f}")
print(f"  Limite superior  : {limite_superior:.2f}")
print("-" * 45)
print(f"  Outliers encontrados : {len(outliers_time)}")
print(f"  % do dataset         : {len(outliers_time)/len(df)*100:.2f}%")
print("=" * 45)

#### Discussão: Outliers e Detecção de Fraude

A análise pelo método IQR identificou outliers nas variáveis Amount e Time.
No contexto de detecção de fraude, a remoção de outliers deve ser feita com
cautela, pelos seguintes motivos:

- **Outliers podem ser fraudes:** transações fraudulentas frequentemente
  apresentam valores atípicos (ex.: valores muito altos ou horários incomuns),
  ou seja, os outliers podem coincidir exatamente com a classe minoritária
  (Class=1) que desejamos detectar.

- **Remover outliers = perder fraudes:** se removermos esses registros,
  podemos estar descartando justamente os casos mais importantes para o
  treinamento do modelo.

Por esses motivos, optamos por **não remover os outliers** neste pipeline,
preservando todas as transações suspeitas para que o modelo possa aprender
a identificá-las corretamente.

***7.Normalização/padronização:***

In [ ]:
scaler = StandardScaler()

# Treina o scaler apenas no treino para evitar data leakage
X_train_scaled = scaler.fit_transform(X_train)

# Aplica a mesma escala no teste, sem reajustar
X_test_scaled = scaler.transform(X_test)

***8.Codificação de variáveis:***

In [ ]:
print("=" * 45)
print("TIPOS DAS VARIÁVEIS")
print("=" * 45)
print(df.dtypes)
print("=" * 45)

# Não foram identificadas variáveis categóricas no conjunto de dados, pois todas as variáveis são numéricas.

***9.Balanceamento de classe:***

In [ ]:
smote = SMOTE(random_state=SEED)

X_train_bal, y_train_bal = smote.fit_resample(
    X_train_scaled,
    y_train
)

antes = y_train.value_counts()
depois = pd.Series(y_train_bal).value_counts()

print("=" * 45)
print("BALANCEAMENTO COM SMOTE")
print("=" * 45)
print(f"{'Classe':<12} {'Antes':>10} {'Depois':>10}")
print("-" * 45)
for classe in sorted(antes.index):
    label = "Legítima (0)" if classe == 0 else "Fraude (1)"
    print(f"{label:<12} {antes[classe]:>10} {depois[classe]:>10}")
print("=" * 45)
print(f"\nAmostras sintéticas geradas: {depois[1] - antes[1]}")
print(f"Total de amostras após SMOTE: {len(y_train_bal)}")

## Treinamento do Modelo:

#### Justificativa do Modelo: Random Forest

O modelo escolhido foi o classificador ***Random Forest***, pelos seguintes motivos:

- **Robusto a outliers:** por ser baseado em árvores de decisão, não é
  sensível a valores extremos como modelos lineares seriam.
- **Não assume distribuição dos dados:** ideal para este dataset, onde
  Amount e Time têm distribuições assimétricas.
- **Bom desempenho em dados desbalanceados:** especialmente após o
  balanceamento com SMOTE, o Random Forest tende a generalizar bem
  para a classe minoritária.
- **Lida bem com muitas variáveis:** com 30 atributos, o ensemble de
  árvores seleciona subconjuntos aleatórios de features, reduzindo
  overfitting.
- **Interpretabilidade parcial:** permite extrair a importância das
  variáveis, útil para análises futuras.

In [ ]:
model = RandomForestClassifier(
    random_state=SEED,
    n_estimators=100, # Número de árvores no ensemble
    n_jobs=-1
)

model.fit(X_train_bal, y_train_bal)

pred = model.predict(X_test_scaled)

## Avaliação:

In [ ]:

# Relatório completo de classificação
print("=" * 55)
print("RELATÓRIO DE CLASSIFICAÇÃO")
print("=" * 55)
print(classification_report(y_test, pred, target_names=["Legítima (0)", "Fraude (1)"]))

# Métricas da classe minoritária (foco principal)
print("=" * 55)
print("MÉTRICAS — CLASSE FRAUDE (1)")
print("=" * 55)
print(f"  Precision : {precision_score(y_test, pred):.4f}")
print(f"  Recall    : {recall_score(y_test, pred):.4f}")
print(f"  F1-Score  : {f1_score(y_test, pred):.4f}")
print(f"  Support   : {int((y_test == 1).sum())} transações fraudulentas no teste")
print("=" * 55)

# Matriz de confusão
cm = confusion_matrix(y_test, pred)

fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Legítima (0)", "Fraude (1)"]
)
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)

# Anotações manuais com rótulos VP/FP/FN/VN
labels = [["VN", "FP"], ["FN", "VP"]]
for i in range(2):
    for j in range(2):
        ax.text(j, i + 0.35, labels[i][j],
                ha="center", va="center",
                fontsize=10, color="gray")

plt.title("Matriz de Confusão", fontsize=13, pad=12)
plt.ylabel("Classe Verdadeira")
plt.xlabel("Classe Predita")
plt.tight_layout()
plt.show()

## Avaliação da Performance do Modelo

O modelo Random Forest foi treinado no conjunto de treino balanceado pelo SMOTE
e avaliado no conjunto de teste com a distribuição original (desbalanceada),
simulando um cenário real de detecção de fraude.

Como a base é fortemente desbalanceada (apenas 0,172% de fraudes), a acurácia
isolada não é uma boa métrica, um modelo que previsse tudo como legítimo
atingiria ~99,8% de acurácia sem detectar nenhuma fraude. Por isso, as métricas
foco são precision, recall e F1 da classe minoritária (Classe 1).

**Resultados para a classe 1 (fraude):**

| Métrica   | Valor  |
|-----------|--------|
| Precision | 0.9125 |
| Recall    | 0.7684 |
| F1-Score  | 0.8343 |

**Matriz de Confusão:**

|                  | Predito: Classe 0 | Predito: Classe 1 |
|------------------|-------------------|-------------------|
| Real: Classe 0   | 56.644 (VP)       | 7 (FP)            |
| Real: Classe 1   | 22 (FN)           | 73 (VP)           |

**Interpretação:**

- O modelo acertou 73 das 95 fraudes presentes no conjunto de teste (**recall de 76,8%**),
  ou seja, deixou passar 22 transações fraudulentas (falsos negativos).

- Das transações classificadas como fraude, 91,25% eram de fato fraudes
  (**precision de 91,25%**), gerando apenas 7 alarmes falsos entre 56.651 transações legítimas.

- O **F1-Score de 0.83** indica um bom equilíbrio entre precision e recall,
  considerando o forte desbalanceamento do dataset.

- A acurácia geral de 99,97% reflete a dominância da classe 0, confirmando que
  ela não deve ser usada como métrica principal neste contexto.

# Questão 2:

## Amostragem Aleatória dos Dados para Clusterização:

In [ ]:
X_cluster = X_train_scaled

amostra = pd.DataFrame(X_cluster).sample(
    frac=0.10,
    random_state=SEED
)

print(amostra.shape)

## Método do Cotovelo:

In [ ]:
sse = []

for k in range(2, 6):

    kmeans = KMeans(
        n_clusters=k,
        random_state=SEED,
        n_init=10
    )

    kmeans.fit(amostra)

    sse.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(2,6), sse, marker='o')
plt.xlabel('Número de clusters (k)')
plt.ylabel('SSE')
plt.title('Método do Cotovelo')

plt.show()

*Escolha do melhor k*

A partir do gráfico do Método do Cotovelo, observa-se que a curva do SSE apresenta uma redução mais acentuada até k=2, após o qual a queda passa a ser menos expressiva, caracterizando o "cotovelo" da curva.

In [ ]:
melhor_k = 2  # escolhido com base no cotovelo identificado no gráfico acima

kmeans = KMeans(
    n_clusters=melhor_k,
    random_state=42,
    n_init=10
)

labels_kmeans = kmeans.fit_predict(amostra)

sse_kmeans = kmeans.inertia_
silhouette_kmeans = silhouette_score(
    amostra,
    labels_kmeans
)

print("SSE:", sse_kmeans)
# Silhouette: mede coesão e separação dos clusters (-1 a 1, quanto maior melhor)
print("Silhouette:", silhouette_kmeans)

## DBSCAN

In [ ]:
dbscan = DBSCAN()

labels_db = dbscan.fit_predict(amostra)

n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_outliers = list(labels_db).count(-1)

print('Clusters:', n_clusters)
print('Outliers:', n_outliers)

if n_clusters > 1:
    sil_dbscan = silhouette_score(amostra, labels_db)
    print('Silhouette:', sil_dbscan)
else:
    sil_dbscan = None

## Avaliação da qualidade: 

In [ ]:
resultados = pd.DataFrame({
    'Algoritmo': ['K-Means', 'DBSCAN'],
    'Nº Clusters': [kmeans.n_clusters if hasattr(kmeans, "n_clusters") else len(set(labels_kmeans)), n_clusters],
    'Outliers': [0, n_outliers],
    'Silhouette': [silhouette_kmeans, sil_dbscan],
    'SSE': [sse_kmeans, None]  # DBSCAN não tem SSE padrão
})

resultados

## Avaliação da Qualidade dos Agrupamentos

**Tabela comparativa:**

| Algoritmo | Nº Clusters | Outliers | Silhouette | SSE           |
|-----------|-------------|----------|------------|---------------|
| K-Means   | 2           | 0        | 0.0664     | 666.439,53    |
| DBSCAN    | 99          | 21.349   | -0.4015    | N/A           |

---

**K-Means(k = 2):**

O método do cotovelo indicou k = 2 como melhor valor, com SSE de **666.439,53**.
O SSE mede a soma das distâncias quadráticas de cada ponto ao centroide do seu
cluster(quanto menor, mais compactos são os grupos).

O coeficiente de silhueta obtido foi de **0.0664**, o que indica que 
os clusters formados têm baixa separação entre si. Isso é esperado
neste dataset: as variáveis V1–V28 são componentes de PCA distribuídos em um
espaço de alta dimensionalidade, sem agrupamentos naturalmente bem delimitados.

---

**DBSCAN(eps = 0.5, min_samples = 5):**

O DBSCAN identificou **99 clusters** e classificou **21.349 pontos como outliers**, o que representa aproximadamente 37% da amostra utilizada.

O coeficiente de silhueta de **-0.4015**, o que indica má qualidade dos
agrupamentos, muitos pontos estão mais próximos de clusters vizinhos do que
do próprio cluster atribuído. Isso sugere que os parâmetros padrão (eps = 0.5,
min_samples = 5) são inadequados para este dataset de alta dimensionalidade,
onde a noção de "vizinhança" é muito mais dispersa.

O SSE não se aplica ao DBSCAN, pois o algoritmo não define centroides.

---

**Conclusão:**

Portanto, considerando as métricas obtidas, o **K-Means com k = 2** apresentou
resultados mais consistentes para este conjunto de dados, sendo também compatível
com a natureza binária do problema original (fraude x legítima). O DBSCAN, por
sua vez, mostrou dificuldades em identificar estruturas de agrupamento adequadas
com os parâmetros padrão, produzindo um número excessivo de clusters e outliers.